## Setting up Environment and PySpark
- ensured the prequisites checked(python,Java and anaconda)
- created an isolated environment and installed all needed libraries including pyspark,jupyter,pandas and mathplolib.

## STEP 2: Create Your First SparkSession

In [17]:
from pyspark.sql import SparkSession
#build sparksession
spark = SparkSession.builder \
.appName("Day4Tutorial") \
.config("spark.sql.shuffle.partitions","4") \
.getOrCreate()

#set log level to reduce noise
spark.sparkContext.setLogLevel("WARN")

print(f"Spark Version: {spark.version}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Master: {spark.sparkContext.master}")

#stop session
spark.stop()

Spark Version: 3.5.1
App Name: RDDTutorial
Master: local[*]


## STEP 3: Working with RDDs

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RDDTutorial") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
sc = spark.sparkContext

# Create an RDD from a Python list
data = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
rdd = sc.parallelize(data)

print("Original RDD:", rdd.collect())

# TRANSFORMATIONS (lazy — nothing happens yet)
squared = rdd.map(lambda x: x ** 2)
evens = rdd.filter(lambda x: x % 2 == 0)
pairs = rdd.map(lambda x: (x % 3, x))

print("\nSquared (after action):", squared.collect())
print("Evens:", evens.collect())
print("Pairs:", pairs.collect())

# ACTIONS (trigger execution)
print("\n--- Actions ---")
print("Count:", rdd.count())
print("Sum:", rdd.reduce(lambda a, b: a + b))
print("Max:", rdd.max())
print("First 3:", rdd.take(3))

# Key-value pair operations
pair_rdd = sc.parallelize([("a", 1), ("b", 2), ("a", 3), ("b", 4), ("c", 5)])
print("\nGroupByKey:", pair_rdd.groupByKey().mapValues(list).collect())
print("ReduceByKey:", pair_rdd.reduceByKey(lambda x, y: x + y).collect())

spark.stop()

Original RDD: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

Squared (after action): [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Evens: [2, 4, 6, 8, 10]
Pairs: [(1, 1), (2, 2), (0, 3), (1, 4), (2, 5), (0, 6), (1, 7), (2, 8), (0, 9), (1, 10)]

--- Actions ---
Count: 10
Sum: 55
Max: 10
First 3: [1, 2, 3]

GroupByKey: [('c', [5]), ('a', [1, 3]), ('b', [2, 4])]
ReduceByKey: [('c', 5), ('a', 4), ('b', 6)]


## STEP 4: Working with DataFrames

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, max, min, count, desc
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder \
    .appName("DataFrameTutorial") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# --- 4.1 Create DataFrame from a list ---
data = [
    ("Alice", "HR", 34, 55000.0),
    ("Bob", "IT", 45, 78000.0),
    ("Charlie", "IT", 29, 62000.0),
    ("Diana", "Finance", 31, 67000.0),
    ("Eve", "HR", 38, 59000.0),
    ("Frank", "Finance", 42, 72000.0),
    ("Grace", "IT", 27, 58000.0),
]

columns = ["Name", "Department", "Age", "Salary"]
df = spark.createDataFrame(data, columns)

print("=== Full DataFrame ===")
df.show()

print("=== Schema ===")
df.printSchema()

# --- 4.2 Select and filter ---
print("=== Select Name and Salary ===")
df.select("Name", "Salary").show()

print("=== Filter: Age > 30 ===")
df.filter(col("Age") > 30).show()

print("=== Filter: IT Department ===")
df.filter(col("Department") == "IT").show()

# --- 4.3 Aggregations ---
print("=== Average Salary by Department ===")
df.groupBy("Department").agg(
    avg("Salary").alias("AvgSalary"),
    max("Salary").alias("MaxSalary"),
    min("Salary").alias("MinSalary"),
    count("*").alias("EmployeeCount")
).orderBy(desc("AvgSalary")).show()

# --- 4.4 Sorting ---
print("=== Sorted by Salary (descending) ===")
df.orderBy(desc("Salary")).show()

# --- 4.5 Adding a computed column ---
from pyspark.sql.functions import when
df_with_bonus = df.withColumn(
    "Bonus",
    when(col("Salary") > 70000, col("Salary") * 0.15)
    .when(col("Salary") > 60000, col("Salary") * 0.10)
    .otherwise(col("Salary") * 0.05)
)
print("=== With Bonus Column ===")
df_with_bonus.show()

# --- 4.6 Spark SQL ---
df.createOrReplaceTempView("employees")

print("=== Spark SQL: Top 3 earners ===")
spark.sql("""
    SELECT Name, Department, Salary
    FROM employees
    ORDER BY Salary DESC
    LIMIT 3
""").show()

print("=== Spark SQL: Department summary ===")
spark.sql("""
    SELECT Department,
           COUNT(*) AS NumEmployees,
           ROUND(AVG(Salary), 2) AS AvgSalary
    FROM employees
    GROUP BY Department
    ORDER BY AvgSalary DESC
""").show()

spark.stop()

=== Full DataFrame ===
+-------+----------+---+-------+
|   Name|Department|Age| Salary|
+-------+----------+---+-------+
|  Alice|        HR| 34|55000.0|
|    Bob|        IT| 45|78000.0|
|Charlie|        IT| 29|62000.0|
|  Diana|   Finance| 31|67000.0|
|    Eve|        HR| 38|59000.0|
|  Frank|   Finance| 42|72000.0|
|  Grace|        IT| 27|58000.0|
+-------+----------+---+-------+

=== Schema ===
root
 |-- Name: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Age: long (nullable = true)
 |-- Salary: double (nullable = true)

=== Select Name and Salary ===
+-------+-------+
|   Name| Salary|
+-------+-------+
|  Alice|55000.0|
|    Bob|78000.0|
|Charlie|62000.0|
|  Diana|67000.0|
|    Eve|59000.0|
|  Frank|72000.0|
|  Grace|58000.0|
+-------+-------+

=== Filter: Age > 30 ===
+-----+----------+---+-------+
| Name|Department|Age| Salary|
+-----+----------+---+-------+
|Alice|        HR| 34|55000.0|
|  Bob|        IT| 45|78000.0|
|Diana|   Finance| 31|67000.0|
| 

## STEP 5: Word Count in PySpark

In [5]:
text = """big data analytics is transforming the world
spark is a fast big data processing engine
spark provides in memory computing for big data
hadoop and spark are key big data technologies
big data requires distributed computing frameworks
spark sql makes big data querying easy
machine learning on big data is powerful
big data and ai are reshaping industries"""

with open("sample.txt", "w") as f:
    f.write(text)

print("sample.txt created")

sample.txt created


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split, col, desc, lower, trim

spark = SparkSession.builder \
    .appName("WordCount") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Read the text file
df = spark.read.text("sample.txt")
print("=== Raw Lines ===")
df.show(truncate=False)

# Split into words and explode
words = df.select(
    explode(split(lower(col("value")), "\\s+")).alias("word")
).filter(trim(col("word")) != "")

# Count words
word_counts = words.groupBy("word").count().orderBy(desc("count"))

print("=== Word Counts ===")
word_counts.show(20)

# Alternative: Using RDD API
rdd = spark.sparkContext.textFile("sample.txt")
rdd_counts = rdd.flatMap(lambda line: line.lower().split()) \
    .map(lambda word: (word, 1)) \
    .reduceByKey(lambda a, b: a + b) \
    .sortBy(lambda x: x[1], ascending=False)

print("=== Word Counts (RDD API) ===")
for word, count in rdd_counts.take(10):
    print(f"{word}: {count}")

spark.stop()

=== Raw Lines ===
+--------------------------------------------------+
|value                                             |
+--------------------------------------------------+
|big data analytics is transforming the world      |
|spark is a fast big data processing engine        |
|spark provides in memory computing for big data   |
|hadoop and spark are key big data technologies    |
|big data requires distributed computing frameworks|
|spark sql makes big data querying easy            |
|machine learning on big data is powerful          |
|big data and ai are reshaping industries          |
+--------------------------------------------------+

=== Word Counts ===
+------------+-----+
|        word|count|
+------------+-----+
|        data|    8|
|         big|    8|
|       spark|    4|
|          is|    3|
|   computing|    2|
|         and|    2|
|         are|    2|
|         for|    1|
|    provides|    1|
|    powerful|    1|
|          in|    1|
|     machine|    1|
| distribu

## STEP 6: Spark SQL and the Catalyst Optimiser

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("CatalystDemo") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Create a larger DataFrame
data = [(i, f"User_{i}", i % 5, i * 100.0) for i in range(1, 1001)]
columns = ["id", "name", "category", "amount"]
df = spark.createDataFrame(data, columns)

# Register as a temporary view
df.createOrReplaceTempView("transactions")

# View the logical plan
print("=== Logical Plan ===")
query = spark.sql("""
    SELECT category, COUNT(*) as cnt, SUM(amount) as total
    FROM transactions
    WHERE amount > 500
    GROUP BY category
    ORDER BY total DESC
""")
query.explain(True)

print("\n=== Results ===")
query.show()

# Compare filter pushdown
print("=== Optimised: Filter before join ===")
df_filtered = df.filter(col("amount") > 500)
df_filtered.groupBy("category").count().show()

spark.stop()

=== Logical Plan ===
== Parsed Logical Plan ==
'Sort ['total DESC NULLS LAST], true
+- 'Aggregate ['category], ['category, 'COUNT(1) AS cnt#247, 'SUM('amount) AS total#248]
   +- 'Filter ('amount > 500)
      +- 'UnresolvedRelation [transactions], [], false

== Analyzed Logical Plan ==
category: bigint, cnt: bigint, total: double
Sort [total#248 DESC NULLS LAST], true
+- Aggregate [category#241L], [category#241L, count(1) AS cnt#247L, sum(amount#242) AS total#248]
   +- Filter (amount#242 > cast(500 as double))
      +- SubqueryAlias transactions
         +- View (`transactions`, [id#239L,name#240,category#241L,amount#242])
            +- LogicalRDD [id#239L, name#240, category#241L, amount#242], false

== Optimized Logical Plan ==
Sort [total#248 DESC NULLS LAST], true
+- Aggregate [category#241L], [category#241L, count(1) AS cnt#247L, sum(amount#242) AS total#248]
   +- Project [category#241L, amount#242]
      +- Filter (isnotnull(amount#242) AND (amount#242 > 500.0))
         +- Lo

# STEP 7: Spark Streaming
## 7.1 Create the Data Generator
created manually and code added as generator.py
## 7.2 Streaming app
created streaming_app.py


### run generator for data source

In [2]:
import subprocess
import sys

generator_process = subprocess.Popen(
    [sys.executable, "generator.py"]
)

print("Generator started")

Generator started


#### run streaming app for data processing in continous form

In [3]:
import subprocess
import sys

spark_process = subprocess.Popen(
    [sys.executable, "streaming_app.py"]
)

print("Spark streaming started")

Spark streaming started


## STEP 8: Machine Learning with MLlib
created step8_mllib.py in separate file and runned below with subprocess

In [2]:
import subprocess
import sys

spark_process = subprocess.Popen(
    [sys.executable, "step8_mllib.py"]
)

print("Machine Learning with MLlib opened, check the anaconda terminal")

Machine Learning with MLlib


## STEP 9: Creating the Dataset for the Exercise
Created create_dataset.py separate file to generate a realistic e-commerce dataset, run it with below cell

In [3]:
import subprocess
import sys

spark_process = subprocess.Popen(
    [sys.executable, "create_dataset.py"]
)

print("a realistic e-comerce dataset created")

a realistic e-comerce dataset created


## STEP 10: Exercise — PySpark Analytics on the E-Commerce Dataset
using ecommerce_sales.csv generated on step 9, wrote PySpark code to answer the execise questions in a file named exercise_solution.py which can be simply runned with codes below

In [5]:
import subprocess
import sys

spark_process = subprocess.Popen(
    [sys.executable, "exercise_solution.py"]
)

print("exercise questions answered, check the running anaconda prompt.")

exercise questions answered, check the running anaconda terminal
